# Homework 2 — CSP: refinamiento y análisis

Modelo: `qwen2.5:3b` · Problemas: 20 · Brazos: A, B, B′ y C.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "results" / "all_arms_combined.csv").exists() and (ROOT / "hw2" / "results" / "all_arms_combined.csv").exists():
    ROOT = ROOT / "hw2"
sys.path.insert(0, str(ROOT / "code"))
sys.path.insert(0, str(ROOT))
from persona3 import bank, evaluate, scorecard, make_figures


## Problemas y modelos A

Dificultad: fácil (≤ 4 variables), media (5–6) y difícil (≥ 7).


In [2]:
problems = bank()
assert len(problems) == 20
for pid in [f"q{i:02}" for i in range(15, 21)]:
    p = problems[pid]
    solved = evaluate(json.dumps(p["spec"], ensure_ascii=False))
    assert solved["signal"] == "none" and solved["answer"] == p["gold"]
print("Problemas:", len(problems))
print("Modelos A q15–q20: 6/6 correctos")
print(pd.Series([v["level"] for v in problems.values()]).value_counts().to_string())


Problemas: 20
Modelos A q15–q20: 6/6 correctos
medium    8
easy      6
hard      6


## Comparación de los cuatro brazos


In [3]:
combined = pd.read_csv(ROOT / "results" / "all_arms_combined.csv")
summary = scorecard(combined)
assert all(summary.set_index("arm").loc[["A", "B", "Bprime", "C"], "n"] == 20)
print(summary.to_string(index=False))


   arm  n  correct  incorrect  no_answer  accuracy  tokens_in  tokens_out  llm_compute_s
     A 20       20          0          0      1.00          0           0       0.000000
     B 20        4          8          8      0.20       6666        4703     209.102530
Bprime 20        4          8          8      0.20      22604       11468     745.122846
     C 20        7         13          0      0.35       3326        1087      49.409756


## Refinamiento B′

Se reutiliza la salida original de B. El modelo recibe únicamente errores del solver, con un máximo de tres correcciones.


In [4]:
bprime = pd.read_csv(ROOT / "results" / "bprime_results.csv")
traces = pd.read_csv(ROOT / "results" / "bprime_traces.csv")
print(bprime[["id", "result", "signal", "iterations"]].to_string(index=False))
print("\nDistribución de correcciones:")
print(bprime["iterations"].value_counts().sort_index().to_string())
print("\nConvergencia de los casos que recibieron corrección:")
refined = bprime[bprime["iterations"] > 0]
print(pd.crosstab(refined["iterations"], refined["converged"]).to_string())


 id    result          signal  iterations
q01   correct            none           0
q02 incorrect            none           0
q03 no_answer           unsat           3
q04 incorrect            none           0
q05 no_answer           unsat           3
q06 incorrect            none           0
q07 no_answer           unsat           3
q08 incorrect            none           0
q09   correct            none           0
q10 incorrect            none           0
q11 no_answer           unsat           3
q12 no_answer           unsat           3
q13 no_answer           unsat           3
q14 no_answer           unsat           3
q15   correct            none           0
q16   correct            none           0
q17 incorrect            none           0
q18 incorrect            none           0
q19 no_answer malformed_model           3
q20 incorrect            none           0

Distribución de correcciones:
iterations
0    12
3     8

Convergencia de los casos que recibieron corrección:
conver

## Costo por respuesta correcta

Costo equivalente de inferencia local: **USD 0,25 por hora de cómputo**, supuesto ilustrativo. No incluye electricidad ni tiempo de desarrollo humano.

Costo por respuesta correcta = (segundos de inferencia ÷ 3600 × tarifa por hora) ÷ respuestas correctas.


In [5]:
tarifa_hora = 0.25
cost = summary[["arm", "correct", "llm_compute_s"]].copy()
cost["estimated_total_usd"] = cost["llm_compute_s"] / 3600 * tarifa_hora
cost["usd_per_correct"] = cost["estimated_total_usd"] / cost["correct"]
print(cost.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


   arm  correct  llm_compute_s  estimated_total_usd  usd_per_correct
     A       20       0.000000             0.000000         0.000000
     B        4     209.102530             0.014521         0.003630
Bprime        4     745.122846             0.051745         0.012936
     C        7      49.409756             0.003431         0.000490


## Figuras


In [6]:
figures = make_figures(combined)
for figure in sorted(figures):
    print(figure.relative_to(ROOT))


fig/accuracy_vs_difficulty.png
fig/arm_accuracy.png
fig/refinement_iterations.png
fig/time_components.png
